# Input-calibration plotter

Compares FC telemetry (MAVSDK Position/Quaternion/Velocity/IMU) and Gazebo
ground-truth poses against the commanded attitude-rate + thrust profile sent
by `input_calibration.py`.

Modeled after `~/ws/scripts/soft_precise_landing/plotter_input_calibration.ipynb`
but reads from `PX4_Gazebo/calibration_data/input/<timestamp>/`.

Frame conventions:
- **W** = Gazebo world (ENU)
- **B** = body FRD (Forward/Right/Down)
- **I** = MAVSDK PositionBody frame (matches FRD)
- `FRD_2_FLU = DCM(x=180°)` converts the rotation-matrix-derived body frame
  (FLU from Gazebo quaternion) into FRD.

In [ ]:
import os, ast
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter as sgf
from ahrs import Quaternion, DCM

np.set_printoptions(precision=2, suppress=True)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# Pick the most recent input-calibration recording, or set RUN_DIR explicitly.
PARENT = '/home/shubham/Soft-Precise-Landing/PX4_Gazebo/calibration_data/input'
_cands = [d for d in os.listdir(PARENT) if os.path.isdir(os.path.join(PARENT, d))]
RUN_DIR = os.path.join(PARENT, max(_cands, key=lambda d: os.path.getmtime(os.path.join(PARENT, d))))
print(f'Loading from: {RUN_DIR}')

tel  = np.load(f'{RUN_DIR}/Telemetry_Data.npy', allow_pickle=True)[()]
gt   = np.load(f'{RUN_DIR}/Ground_Truth.npy',   allow_pickle=True)[()]
try:
    img = np.load(f'{RUN_DIR}/Img_Data.npy', allow_pickle=True)[()]
except FileNotFoundError:
    img = None

print('tel keys:', list(tel.keys()))
print('gt  keys:', list(gt.keys()))
print(f'gt samples: {len(gt["Time"])}, duration {gt["Time"][-1] - gt["Time"][0]:.2f}s')

## Frame conventions

In [ ]:
FRD_2_FLU = np.array(DCM(x=180.0))
mass = 2.114   # kg, Holybro X500 (matches MATLAB Constants.m)
g_scalar = 9.8

## Telemetry data (MAVSDK)

PositionBody / Quaternion / VelocityBody / AngularVelocityBody at odometry rate (~60 Hz).

In [ ]:
start_idx_o = np.searchsorted(tel['Odometry Timestamp'], gt['Start Time'])
positions     = tel['Position Body'][start_idx_o:]
quaternions   = tel['Quaternion'][start_idx_o:]
velocities    = tel['Velocity Body'][start_idx_o:]
ang_vels      = tel['Angular Velocity Body'][start_idx_o:]
t_o = np.array(tel['Odometry Timestamp'][start_idx_o:]) - gt['Start Time']

n_o = len(positions)
B_x_ut = np.zeros((n_o, 3))   # UAV position (FRD body frame)
EA_t   = np.zeros((n_o, 3))   # Euler angles (FRD)
I_R_B  = np.zeros((n_o, 3, 3))
for i, (pos, q) in enumerate(zip(positions, quaternions)):
    I_x_u = np.array([pos.x_m, pos.y_m, pos.z_m])
    I_R_B[i] = Quaternion([q.w, q.x, q.y, q.z]).to_DCM()
    EA_t[i]  = np.linalg.inv(FRD_2_FLU) @ Quaternion([q.w, q.x, q.y, q.z]).to_angles()
    B_x_ut[i] = np.linalg.inv(I_R_B[i]) @ I_x_u

B_v_ut = np.zeros((n_o, 3))
B_w_ut = np.zeros((n_o, 3))
for i, (R, vel, w) in enumerate(zip(I_R_B, velocities, ang_vels)):
    B_v_ut[i] = (np.linalg.inv(R) @ np.array([[vel.x_m_s, vel.y_m_s, vel.z_m_s]]).T).T
    B_w_ut[i] = np.array([w.roll_rad_s, w.pitch_rad_s, w.yaw_rad_s])

print(f'telemetry samples: {n_o}, t in [{t_o[0]:.2f}, {t_o[-1]:.2f}]s')

## IMU data

Acceleration (FRD: forward/right/down) and angular velocity (FRD) — these come directly from PX4's IMU; useful for cross-checking and for computing thrust from `T = m*(a_z + g)`.

In [ ]:
start_idx_a = np.searchsorted(tel['IMU Timestamp'], gt['Start Time'])
t_a = np.array(tel['IMU Timestamp'][start_idx_a:]) - gt['Start Time']
a_t = np.array([[a.forward_m_s2, a.right_m_s2, a.down_m_s2]
                for a in tel['Acceleration'][start_idx_a:]])
w_t = np.array([[w.forward_rad_s, w.right_rad_s, w.down_rad_s]
                for w in tel['Angular Velocity FRD'][start_idx_a:]])
# Thrust from IMU: T = m*(a_down + g)
T_t = mass * (a_t[:, 2] + g_scalar)
print(f'IMU samples: {len(t_a)}')

## Ground truth (Gazebo `/pose`)

Compute UAV pose in body frame, then differentiate world-frame position twice for velocity and acceleration. Heavy savgol smoothing on the derivatives (typical sgf(301, 2) for v, sgf(301, 2) for a — matches the reference notebook).

In [ ]:
t_g = np.array(gt['Time'])
pose0 = gt['Start Pose']
EA_g0 = Quaternion([pose0.orientation.w, pose0.orientation.x,
                    pose0.orientation.y, pose0.orientation.z]).to_angles()

uav_poses = gt['UAV Pose']
n_g = len(uav_poses)
W_R_B = np.zeros((n_g, 3, 3))
W_x_u = np.zeros((n_g, 3))
EA_g  = np.zeros((n_g, 3))
for i, p in enumerate(uav_poses):
    q = Quaternion([p.orientation.w, p.orientation.x, p.orientation.y, p.orientation.z])
    W_R_B[i] = q.to_DCM() @ FRD_2_FLU
    W_x_u[i] = np.array([p.position.x, p.position.y, p.position.z])
    EA_g[i]  = q.to_angles()
    EA_g[i, 2] -= EA_g0[2] - EA_t[0, 2]   # align yaw zero between GT and telemetry

W_v_u = np.gradient(W_x_u, t_g, axis=0)
W_a_u = np.gradient(W_v_u, t_g, axis=0)
for arr in (W_v_u, W_a_u):
    nans, idx = np.isnan(arr), lambda z: z.nonzero()[0]
    arr[nans] = np.interp(idx(nans), idx(~nans), arr[~nans])

B_x_ug_raw = np.zeros((n_g, 3))
B_v_ug_raw = np.zeros((n_g, 3))
B_a_ug_raw = np.zeros((n_g, 3))
for i, (R, x, v, a) in enumerate(zip(W_R_B, W_x_u, W_v_u, W_a_u)):
    B_x_ug_raw[i] = np.linalg.inv(R) @ x
    B_v_ug_raw[i] = np.linalg.inv(R) @ v
    B_a_ug_raw[i] = np.linalg.inv(R) @ a

B_x_ug = B_x_ug_raw
# Heavy smoothing on derivatives (matches reference notebook)
B_v_ug = sgf(B_v_ug_raw, min(301, 2 * (n_g // 4) + 1), 2, axis=0) if n_g >= 11 else B_v_ug_raw
B_a_ug = sgf(B_a_ug_raw, min(301, 2 * (n_g // 4) + 1), 2, axis=0) if n_g >= 11 else B_a_ug_raw

In [ ]:
# Body angular velocity from time derivative of rotation matrix
W_dR_B = np.gradient(W_R_B, t_g, axis=0)
B_w_ug = np.zeros((n_g, 3))
for i, (R, dR) in enumerate(zip(W_R_B, W_dR_B)):
    skew = R.T @ dR
    B_w_ug[i] = [skew[2, 1], skew[0, 2], skew[1, 0]]
nans, idx = np.isnan(B_w_ug), lambda z: z.nonzero()[0]
B_w_ug[nans] = np.interp(idx(nans), idx(~nans), B_w_ug[~nans])
B_w_ug = sgf(B_w_ug, min(101, 2 * (n_g // 4) + 1), 5, axis=0) if n_g >= 11 else B_w_ug

## Input commands

`gt['Command']` has shape `(N, 4)` — rows are `[roll_rate, pitch_rate, yaw_rate, thrust_delta_N]`.
The thrust delta enters `convert_2_sys_cmd` as `throttle = 0.738 - thrust/45`.

In [ ]:
cmds = np.array(gt['Command'])
w_u = cmds[:, :3]   # commanded rate (rad/s)
T_u = cmds[:, 3]    # commanded thrust delta (N)
print(f'unique commands:\n{np.unique(np.round(cmds, 3), axis=0)}')

## Plots

Each subplot: telemetry vs ground truth (and command where applicable).

### Position (body frame)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, ['x', 'y', 'z'])):
    ax.plot(t_o, B_x_ut[:, i], label='Telemetry')
    ax.plot(t_g, B_x_ug[:, i], label='Ground Truth')
    ax.set(title=f'Position {name} (body, m)', xlabel='t (s)', ylabel=f'{name} (m)')
    ax.legend()
plt.show()

### Euler angles

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, ['roll', 'pitch', 'yaw'])):
    ax.plot(t_o, EA_t[:, i], label='Telemetry')
    ax.plot(t_g, EA_g[:, i], label='Ground Truth')
    ax.set(title=f'{name} (rad)', xlabel='t (s)', ylabel=f'{name} (rad)')
    ax.legend()
plt.show()

### Linear velocity (body frame)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, ['vx', 'vy', 'vz'])):
    ax.plot(t_o, B_v_ut[:, i], label='Telemetry')
    ax.plot(t_g, B_v_ug[:, i], label='Ground Truth')
    ax.set(title=f'Velocity {name} (body, m/s)', xlabel='t (s)', ylabel=f'{name} (m/s)')
    ax.legend()
plt.show()

### Angular velocity — telemetry IMU vs command

The most direct input-calibration plot: how does PX4's measured body rate (IMU) compare to the commanded rate?

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, ['wx', 'wy', 'wz'])):
    ax.plot(t_a, w_t[:, i], label='Telemetry IMU')
    ax.plot(t_g, w_u[:, i], label='Commanded rate', linestyle='--', linewidth=2)
    ax.set(title=f'Angular velocity {name} (body, rad/s)',
           xlabel='t (s)', ylabel=f'{name} (rad/s)')
    ax.legend()
plt.show()

### Linear acceleration

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
g_vec = [0, 0, g_scalar]
for i, (ax, name) in enumerate(zip(axes, ['ax', 'ay', 'az'])):
    # IMU acceleration in body FRD, with gravity removed
    ax.plot(t_a, -a_t[:, i] - g_vec[i], label='Telemetry IMU (g-removed)')
    ax.plot(t_g, B_a_ug[:, i], label='Ground Truth (d²x/dt²)')
    if i == 2:
        # Thrust component on z: T/m maps to -a_z in body FRD
        ax.plot(t_g, -T_u / mass, label='Commanded thrust / m', linestyle='--')
    ax.set(title=f'Acceleration {name} (body, m/s²)',
           xlabel='t (s)', ylabel=f'{name} (m/s²)')
    ax.legend()
plt.show()

### Body thrust

`T_t = mass * (a_down + g)` from IMU vs `T_u` commanded (delta over hover; total throttle in `convert_2_sys_cmd` is `0.738 − T_u/45`).

In [ ]:
plt.figure(figsize=(12, 5), constrained_layout=True)
plt.plot(t_a, T_t, label='Telemetry (m·(a_z + g))')
plt.plot(t_g, T_u, label='Commanded thrust delta (N)', linestyle='--')
plt.title('Body thrust (N)')
plt.xlabel('t (s)'); plt.ylabel('T (N)')
plt.legend(); plt.show()

## Command → response quality

Per-axis Pearson correlation between commanded rate and IMU-measured rate, computed only where the command is non-zero (so idle hover periods don't dominate the metric).

In [ ]:
# Resample command onto IMU timeline
w_u_on_imu = np.array([np.interp(t_a, t_g, w_u[:, i]) for i in range(3)]).T
T_u_on_imu = np.interp(t_a, t_g, T_u)

def corr_active(meas, cmd, threshold):
    m = np.abs(cmd) > threshold
    if m.sum() < 10: return float('nan')
    c = np.corrcoef(meas[m], cmd[m])[0, 1]
    return c

print('Angular-rate command vs IMU measurement (only when |cmd| > 0.01 rad/s):')
for i, name in enumerate(['wx', 'wy', 'wz']):
    c = corr_active(w_t[:, i], w_u_on_imu[:, i], 0.01)
    rmse = np.sqrt(np.nanmean((w_t[:, i] - w_u_on_imu[:, i])**2))
    print(f'  {name}: corr={c:+.3f}  RMSE={rmse:.4f} rad/s')

print('\nThrust command vs IMU-derived (only when |cmd| > 0.1 N):')
c = corr_active(T_t, T_u_on_imu, 0.1)
rmse = np.sqrt(np.nanmean((T_t - T_u_on_imu)**2))
print(f'  T: corr={c:+.3f}  RMSE={rmse:.4f} N   (hover T_t ≈ {mass*g_scalar:.2f} N)')